## Exercise 14.4 (Parallel coordinates plots)  

Use parallel coordinates plots to visualize the relationships between multiple features and the machine failure types:

1. Display every available continuous variable along with one selected failure type.
2. Also add Type as a categorical variable to the plot.
3. Make the failure type a categorical column.
4. Color the lines according to Type.
5. Allow for choosing the failure type interactively using a dropdown widget (see Jupyter widgets) with the available failure types.
6. The parallel coordinates plot allows for dragging columns (just click and dag the title of a column) and reordering them. It also allows for selecting and highlighting specific lines, making it easier to focus on particular data points. Use these two features to get insights into the relationships between different features and the selected failure type. Can you identify the patterns that we exploited in our previous analyses?

In [1]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Datensatz laden
df = pd.read_csv(
    "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"
)

# Ausgabe der ersten Zeilen des DataFrames
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


1. Display every available continuous variable along with one selected failure type.
2. Also add Type as a categorical variable to the plot.
3. Make the failure type a categorical column.
4. Color the lines according to Type.
5. Allow for choosing the failure type interactively using a dropdown widget (see Jupyter widgets) with the available failure types.

In [2]:
# Spalten und Kategorien definieren
# Spaltennamen
continuous_vars = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]
# Error-Variablen
error_vars = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]

# Produktvariante als Kategorische Variable definieren
df["Type"] = pd.Categorical(
    df["Type"],
    categories=["L", "M", "H"]
)

# Fehlerindikatoren als Kategorische Variablen definieren
for error in error_vars:
    df[error] = pd.Categorical(
        df[error],
        categories=[0, 1]
    )

In [ ]:
# Parallelkoordinatendiagramm erstellen
def plot_parallel(failure):
    dimensions = []

    # Eine Achse für jede Messgröße erstellen
    for variable in continuous_vars:
        dimensions.append({
            "label": variable,
            "values": df[variable]
        })

    # Produktvariante und Fehlerindikator als Achsen hinzufügen
    dimensions.append({
        "label": "Type",
        "values": df["Type"].cat.codes,
        "tickvals": [0, 1, 2],
        "ticktext": ["L", "M", "H"],
        "range": [0, 2]
    })

    # Ausgewählte Fehlerart als Achse hinzufügen
    dimensions.append({
        "label": failure,
        "values": df[failure].cat.codes,
        "tickvals": [0, 1],
        "ticktext": ["kein Fehler", "Fehler"],
        "range": [0, 1]
    })

    fig = go.Figure(
        go.Parcoords(
            dimensions=dimensions,
            line={
                "color": df["Type"].cat.codes,
                "cmin": 0,
                "cmax": 2,
                "colorscale": [
                    [0.0, "steelblue"],
                    [0.25, "steelblue"],
                    [0.25, "orange"],
                    [0.75, "orange"],
                    [0.75, "seagreen"],
                    [1.0, "seagreen"]
                ],
                "colorbar": {
                    "title": "Type",
                    "tickvals": [0, 1, 2],
                    "ticktext": ["L", "M", "H"]
                }
            }
        )
    )

    fig.update_layout(
        title=f"Parallelkoordinatendiagramm: {failure}",
        height=550,
        margin=dict(l=60, r=80, t=100, b=50)
    )

    fig.show()

# Interaktive Dropdown-Auswahl für die Fehlerart
dropdown = widgets.Dropdown(
    options=error_vars,
    value="OSF",
    description="Fehlerart:"
)

# Dropdown-Widget und interaktive Ausgabe verbinden
output = widgets.interactive_output(
    plot_parallel,
    {"failure": dropdown}
)

display(dropdown, output)

Dropdown(description='Fehlerart:', index=4, options=('Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'), va…

Output()

6. The parallel coordinates plot allows for dragging columns (just click and dag the title of a column) and reordering them. It also allows for selecting and highlighting specific lines, making it easier to focus on particular data points. Use these two features to get insights into the relationships between different features and the selected failure type. Can you identify the patterns that we exploited in our previous analyses?

**Auswertung:**
- **TWF:** Die hervorgehobenen Beobachtungen weisen überwiegend einen hohen Werkzeugverschleiß ab etwa 200 Minuten auf. Die übrigen Messgrößen sind breiter verteilt. Damit lässt sich der bekannte Zusammenhang zwischen Werkzeugverschleiß und TWF wiedererkennen.
- **HDF:** Die Fehlerfälle konzentrieren sich auf niedrige Drehzahlen von ungefähr 1.200 bis 1.400 rpm. Die Umgebungstemperatur liegt überwiegend im oberen Bereich. Die aus der bisherigen Analyse bekannte kleine Temperaturdifferenz ist hier schwerer zu erkennen, weil beide Temperaturachsen unterschiedliche Skalen besitzen.
- **PWF:** Zwischen Drehzahl und Drehmoment sind zwei Gruppen sichtbar: hohe Drehzahlen mit niedrigem Drehmoment sowie niedrige Drehzahlen mit hohem Drehmoment. Das passt zu den unterschiedlichen Kombinationen, die zu Leistungsfehlern führen. Die genauen Leistungsgrenzen lassen sich ohne eine eigene Leistungsachse jedoch nicht direkt ablesen.
- **OSF:** Die Fehlerfälle zeigen überwiegend eine Kombination aus hohem Werkzeugverschleiß und hohem Drehmoment. Damit ist der bekannte Zusammenhang wiedererkennbar. Die unterschiedlichen Grenzwerte der Produktvarianten sind anhand der Farben allein nicht eindeutig bestimmbar.
- **RNF:** Die markierten Beobachtungen verteilen sich über verschiedene Wertebereiche. Ein eindeutiges gemeinsames Muster ist nicht erkennbar. Das ist mit der Beschreibung als zufällige Fehler vereinbar.

Die Darstellung hilft, mehrere Merkmale gleichzeitig zu vergleichen. Besonders gut erkennbar sind auffällige Wertebereiche einzelner Merkmale und Kombinationen benachbarter Achsen. Zusammenhänge über berechnete Größen wie Temperaturdifferenz und Leistung sind dagegen weniger deutlich als in den vorherigen Analysen.


Die beobachteten Muster stimmen grundsätzlich mit den bisherigen Ergebnissen überein. Besonders deutlich sind hoher Werkzeugverschleiß bei TWF, niedrige Drehzahlen bei HDF sowie die Kombination aus hohem Verschleiß und hohem Drehmoment bei OSF. Bei RNF ist weiterhin kein klares Muster erkennbar. Die genauen Regeln für Temperaturdifferenz, Leistung und produktspezifische Belastungsgrenzen lassen sich im Parallelkoordinatendiagramm jedoch nicht direkt überprüfen. Dafür waren die berechneten Merkmale und Entscheidungsbäume besser geeignet.

HTML erstellt mit:

uv run jupyter nbconvert --to html "Tasks/14 Self study - Session 1/Exercise_14.4.ipynb"